# BraTS-METs Evaluation Example

This notebook provides a step-by-step example of how to use the `brats-evaluate` and `brats-parse-metrics` console scripts (installed by the `BraTS-evaluation` package) to evaluate brain tumor segmentations.

The sample data used here is from the training set of BraTS 2025 Brain Metastasis (METS) challenge. It includes four cases, each with a reference (ground truth) and a corresponding prediction. While this example
focuses on METS, the same evaluation pipeline can be adapted for other segmentation tasks like glioma, meningioma, or pediatric tumors by choosing the appropriate `--config` name (e.g., `gli`, `ped`, `MenRT`, `MenPre`, `GoAT`).

## Sample Data Structure
The data is organized as follows (pay attention to the naming convention for the predicted masks required by the BraTS challege ):

```
├── pred
│   ├── 00132-000.nii.gz
│   ├── BraTS-MET-00630-001.nii.gz
│   ├── BraTS-MET_01325-000.nii.gz
│   └── SEG-00217-000.nii.gz
└── ref
    ├── BraTS-MET-00132-000-seg.nii.gz
    ├── BraTS-MET-00217-000-seg.nii.gz
    ├── BraTS-MET-00630-001-seg.nii.gz
    └── BraTS-MET-01325-000-seg.nii.gz


```


### 1. Run the Evaluation

First, we call `brats-evaluate` to compare the predictions against the references. This command will generate a detailed JSON file containing all the raw metrics from Panoptica.

In [1]:
!python ../brats_evaluation/evaluation.py \
    --config mets \
    --ref_path ./sample_data/ref/ \
    --pred_path ./sample_data/pred/ \
    --summary_json ./sample_mets_metrics.json

Initializing Panoptica Evaluator...
The same labels [1, 3] were assigned to two different labelgroups, got SegmentationClassGroups = 
 - et : LabelGroup [3], single_instance=False
 - rc : LabelGroup [4], single_instance=False
 - tc : LabelMergeGroup [1, 3] -> ONE, single_instance=False
 - wt : LabelMergeGroup [1, 2, 3] -> ONE, single_instance=False
Intended? This will evaluate the duplicate labels in both groups
Starting evaluation for 4 subjects...
[1/4] Evaluating BraTS-MET-00132-000-seg.nii.gz...
──────────────────────── Thank you for using panoptica ─────────────────────────
                    Please support our development by citing                    
        https://github.com/BrainLesion/panoptica#citation -- Thank you!         
────────────────────────────────────────────────────────────────────────────────

/home/mehdi/anaconda3/envs/bratseval/lib/python3.10/site-packages/panoptica/metrics/normalized_surface_dice.py:41: UserWarning: The threshold is set to 0.5, which is the 

### 2. Inspect the JSON Output

The generated JSON file is highly detailed. Let's load it and print the metrics for the first subject to see the rich, instance-wise information Panoptica provides.

In [2]:
import json

with open('sample_mets_metrics.json', 'r') as f:
    data = json.load(f)
# Pretty-print the first subject's metrics,
print(json.dumps(data['metrics'][0], indent=4))

{
    "et": {
        "n_ref_instances": 8,
        "n_pred_instances": 7,
        "tp": 6,
        "fp": 1,
        "fn": 2,
        "prec": 0.8571428571428571,
        "rec": 0.75,
        "rq": 0.8,
        "sq_dsc": 0.7735988324659234,
        "sq_dsc_std": 0.05146092409268618,
        "pq_dsc": 0.6188790659727388,
        "sq_hd95": 2.0748032315081146,
        "sq_hd95_std": 1.0356253343863966,
        "sq_nsd": 0.8605908094646745,
        "sq_nsd_std": 0.15545837198420034,
        "instance_voxel_count_ref": 916.1666666666666,
        "instance_volume_ref": 916.1666666666666,
        "global_bin_volume_pred": 4800,
        "global_bin_volume_ref": 6915,
        "global_bin_dsc": 0.6840802390098165,
        "global_bin_hd95": 33.52610922848042,
        "global_bin_nsd": 0.32470764194847074,
        "reference_instances": [
            {
                "sq_dsc": 0.7091442282058118,
                "sq_hd95": 3.1622776601683795,
                "sq_nsd": 0.5571437139665475,
       

### 3. Parse the Results of Metastasis Task Into a CSV

Next, we use the `brats-parse-metrics` command with the `mets` subcommand to process the raw JSON file.
This script calculates size-stratified statistics and organizes the results into a clean, human-readable CSV file.

Note: For BraTS 2026 METs challenge, lesions which have volume below `27mm^3` will be considered for the detection evaluation and the overlapping threshold for detection criteria is `DSC=0.2`

In [3]:
!python ../brats_evaluation/metrics_parser.py mets \
    --json_path ./sample_mets_metrics.json \
    --vol_threshold 27 \
    --overlap_threshold 0.2 \
    --output_csv_path ./sample_mets_summary.csv

JSON file loaded successfully.
DataFrame saved successfully to ./sample_mets_summary.csv
                       subject_id  ...  small_instance_f1_wt
2  BraTS-MET-00630-001-seg.nii.gz  ...              1.000000
3  BraTS-MET-01325-000-seg.nii.gz  ...                   NaN
4                            mean  ...              0.571429
5                             std  ...              0.606092
6                          median  ...              0.571429

[5 rows x 73 columns]


### 4. Display the Final CSV Report

Finally, let's load and display the generated CSV file. This table provides a clear summary of the evaluation, making it easy to compare performance across different subjects and metrics.

In [8]:
import pandas as pd
df = pd.read_csv('sample_mets_summary.csv')
display(df)

,subject_id,all_instance_tp_et,all_instance_fp_et,all_instance_fn_et,all_instance_f1_et,large_instance_tp_et,large_instance_fp_et,large_instance_fn_et,large_instance_f1_et,lesionwise_dsc_mean_et,...,lesionwise_dsc_mean_wt,lesionwise_dsc_std_wt,lesionwise_hd95_mean_wt,lesionwise_hd95_std_wt,lesionwise_nsd_mean_wt,lesionwise_nsd_std_wt,small_instance_tp_wt,small_instance_fn_wt,small_instance_fp_wt,small_instance_f1_wt
0,BraTS-MET-00132-000-seg.nii.gz,6.000000,1.000000,2.000000,0.800000,6.000000,1.000000,1.000000,0.857143,0.580199,...,0.518309,0.331111,95.244086,160.365900,0.581402,0.357495,NaN,NaN,NaN,NaN
1,BraTS-MET-00217-000-seg.nii.gz,27.000000,7.000000,7.000000,0.794118,24.000000,7.000000,2.000000,0.842105,0.529192,...,0.317559,0.350759,189.413466,183.667369,0.353160,0.376406,1.0,3.00000,9.000000,0.142857
2,BraTS-MET-00630-001-seg.nii.gz,12.000000,0.000000,0.000000,1.000000,10.000000,0.000000,0.000000,1.000000,0.876783,...,0.858429,0.060251,1.182275,0.400099,0.933376,0.057534,1.0,0.00000,0.000000,1.000000
3,BraTS-MET-01325-000-seg.nii.gz,1.000000,0.000000,1.000000,0.666667,1.000000,0.000000,0.000000,1.000000,0.973133,...,0.985481,0.000000,0.500000,0.000000,0.963291,0.000000,NaN,NaN,NaN,NaN
4,mean,11.500000,2.000000,2.500000,0.815196,10.250000,2.000000,0.750000,0.924812,0.739827,...,0.669944,0.185530,71.584957,86.108342,0.707807,0.197859,1.0,1.50000,4.500000,0.571429
5,std,11.269428,3.366502,3.109126,0.137706,9.878428,3.366502,0.957427,0.087036,0.218355,...,0.306735,0.181302,90.282742,99.653564,0.293204,0.196810,0.0,2.12132,6.363961,0.606092
6,median,9.000000,0.500000,1.500000,0.797059,8.000000,0.500000,0.500000,0.928571,0.728491,...,0.688369,0.195681,48.213181,80.383000,0.757389,0.207515,1.0,1.50000,4.500000,0.571429


The table contains the following metrics:
- **Detection Metrics for All lesion**
- **Detection Metrics for Large Lesions**
- **Lesion-wise Segmentation Metrics for Large Lesions**
- **Detection Metrics for Small Lesions**

As shown above, the last three rows of the DataFrame conveniently provide the **mean** and **standard deviation (std)** and **median** for each metric column, calculated over all valid (non-NaN) rows. This gives a quick statistical overview of the algorithm's performance across the entire dataset."

### 4. Parse the Results of Other Segmentation Tasks Into a CSV

Next, we use the `brats-parse-metrics` command with the `seg` subcommand to process the raw JSON file.
This script parses statistics and organizes the results into a clean, human-readable CSV file.

In [9]:
# For simplicity we use the metrics cmoputed from the METs samples:
!python ../brats_evaluation/metrics_parser.py seg \
    --json_path ./sample_mets_metrics.json \
    --output_csv_path ./sample_seg_summary.csv
df = pd.read_csv('sample_seg_summary.csv')
display(df)

JSON file loaded successfully.
DataFrame saved successfully to ./sample_seg_summary.csv
                       subject_id  ...  global_hd95_wt
2  BraTS-MET-00630-001-seg.nii.gz  ...        1.414214
3  BraTS-MET-01325-000-seg.nii.gz  ...        1.000000
4                            mean  ...        9.814395
5                             std  ...       14.795389
6                          median  ...        3.207107

[5 rows x 29 columns]


,subject_id,all_instance_tp_et,all_instance_fp_et,all_instance_fn_et,all_instance_f1_et,global_dsc_et,global_nsd_et,global_hd95_et,all_instance_tp_rc,all_instance_fp_rc,...,global_dsc_tc,global_nsd_tc,global_hd95_tc,all_instance_tp_wt,all_instance_fp_wt,all_instance_fn_wt,all_instance_f1_wt,global_dsc_wt,global_nsd_wt,global_hd95_wt
0,BraTS-MET-00132-000-seg.nii.gz,6.000000,1.000000,2.000000,0.800000,0.684080,0.324708,33.526109,0.0,0.0,...,0.688781,0.326654,33.541020,6.000000,1.000000,1.000000,0.857143,0.756478,0.277575,31.843367
1,BraTS-MET-00217-000-seg.nii.gz,27.000000,7.000000,7.000000,0.794118,0.791323,0.473528,2.828427,0.0,0.0,...,0.896435,0.451112,2.236068,14.000000,9.000000,7.000000,0.636364,0.826016,0.242283,5.000000
2,BraTS-MET-00630-001-seg.nii.gz,12.000000,0.000000,0.000000,1.000000,0.914795,0.667265,1.000000,0.0,0.0,...,0.926660,0.667764,1.000000,11.000000,0.000000,0.000000,1.000000,0.927545,0.614201,1.414214
3,BraTS-MET-01325-000-seg.nii.gz,1.000000,0.000000,1.000000,0.666667,0.972161,0.763323,1.000000,0.0,0.0,...,0.975164,0.786067,1.000000,1.000000,0.000000,0.000000,1.000000,0.985481,0.697099,1.000000
4,mean,11.500000,2.000000,2.500000,0.815196,0.840590,0.557206,9.588634,0.0,0.0,...,0.871760,0.557899,9.444272,8.000000,2.500000,2.000000,0.873377,0.873880,0.457790,9.814395
5,std,11.269428,3.366502,3.109126,0.137706,0.128763,0.196346,15.981577,0.0,0.0,...,0.126223,0.207371,16.075063,5.715476,4.358899,3.366502,0.171761,0.102321,0.231411,14.795389
6,median,9.000000,0.500000,1.500000,0.797059,0.853059,0.570396,1.914214,0.0,0.0,...,0.911547,0.559438,1.618034,8.500000,0.500000,0.500000,0.928571,0.876780,0.445888,3.207107


This table contains the following metrics:
- **Detection Metrics for All lesion**
- **Global Segmentation Metrics for All Lesions**